In [0]:
flights_data = [
("F101","Indigo","Hyderabad","Delhi",140,"On Time"),
("F102","Air India","Mumbai","Chennai",120,"Delayed"),
("F103","Vistara","Bangalore","Hyderabad",90,"On Time"),
("F104","Indigo","Delhi","Mumbai",130,"Cancelled"),
("F105","Air India","Chennai","Bangalore",80,"On Time"),
("F106","Akasa","Pune","Delhi",150,"Delayed"),
("F107","Vistara","Hyderabad","Kolkata",160,"On Time"),
("F108","Indigo","Mumbai","Hyderabad",110,"On Time"),
("F109","Akasa","Delhi","Chennai",145,"Delayed"),
("F110","Air India","Bangalore","Mumbai",95,"On Time"),
("F111","Indigo","Hyderabad","Goa",75,"On Time"),
("F112","Vistara","Goa","Delhi",150,"Cancelled"),
("F113","Akasa","Chennai","Pune",100,"On Time"),
("F114","Air India","Kolkata","Bangalore",170,"Delayed"),
("F115","Indigo","Delhi","Hyderabad",135,"On Time")
]

columns = [
    "flight_id",
    "airline",
    "from_city",
    "to_city",
    "duration",
    "status"
]

df_flights = spark.createDataFrame(flights_data, columns)

display(df_flights)

flight_id,airline,from_city,to_city,duration,status
F101,Indigo,Hyderabad,Delhi,140,On Time
F102,Air India,Mumbai,Chennai,120,Delayed
F103,Vistara,Bangalore,Hyderabad,90,On Time
F104,Indigo,Delhi,Mumbai,130,Cancelled
F105,Air India,Chennai,Bangalore,80,On Time
F106,Akasa,Pune,Delhi,150,Delayed
F107,Vistara,Hyderabad,Kolkata,160,On Time
F108,Indigo,Mumbai,Hyderabad,110,On Time
F109,Akasa,Delhi,Chennai,145,Delayed
F110,Air India,Bangalore,Mumbai,95,On Time


In [0]:
# ==========================================
# Create Booking DataFrame
# ==========================================

bookings_data = [
("B1001","F101","Rahul Sharma","Economy",8500,"2026-06-01"),
("B1002","F101","Priya Reddy","Business",22000,"2026-06-01"),
("B1003","F102","Amit Kumar","Economy",9000,"2026-06-02"),
("B1004","F103","Sneha Patel","Premium Economy",15000,"2026-06-02"),
("B1005","F104","Farhan Ali","Economy",7500,"2026-06-03"),
("B1006","F105","Neha Singh","Business",25000,"2026-06-03"),
("B1007","F106","Arjun Verma","Economy",10000,"2026-06-04"),
("B1008","F107","Meera Nair","Premium Economy",17000,"2026-06-04"),
("B1009","F108","Kiran Rao","Economy",9500,"2026-06-05"),
("B1010","F109","Nisha Reddy","Business",28000,"2026-06-05"),
("B1011","F110","David Thomas","Economy",8000,"2026-06-06"),
("B1012","F111","Ayesha Khan","Premium Economy",16000,"2026-06-06"),
("B1013","F112","Rohit Sharma","Economy",7000,"2026-06-07"),
("B1014","F113","Pooja Mehta","Business",24000,"2026-06-07"),
("B1015","F114","Sanjay Gupta","Economy",10500,"2026-06-08"),
("B1016","F115","Divya Iyer","Premium Economy",18000,"2026-06-08"),
("B1017","F101","Vikram Singh","Economy",8500,"2026-06-09"),
("B1018","F103","Anjali Rao","Business",23000,"2026-06-09"),
("B1019","F107","Faiz Ahmed","Economy",9500,"2026-06-10"),
("B1020","F110","Megha Kapoor","Premium Economy",15500,"2026-06-10")
]

columns = [
    "booking_id",
    "flight_id",
    "passenger_name",
    "travel_class",
    "ticket_price",
    "booking_date"
]

df_bookings = spark.createDataFrame(
    bookings_data,
    columns
)

In [0]:
joined_df = df_bookings.join(
    df_flights,
    on="flight_id",
    how="inner"
)

display(joined_df)

flight_id,booking_id,passenger_name,travel_class,ticket_price,booking_date,airline,from_city,to_city,duration,status
F101,B1001,Rahul Sharma,Economy,8500,2026-06-01,Indigo,Hyderabad,Delhi,140,On Time
F101,B1002,Priya Reddy,Business,22000,2026-06-01,Indigo,Hyderabad,Delhi,140,On Time
F102,B1003,Amit Kumar,Economy,9000,2026-06-02,Air India,Mumbai,Chennai,120,Delayed
F103,B1004,Sneha Patel,Premium Economy,15000,2026-06-02,Vistara,Bangalore,Hyderabad,90,On Time
F104,B1005,Farhan Ali,Economy,7500,2026-06-03,Indigo,Delhi,Mumbai,130,Cancelled
F105,B1006,Neha Singh,Business,25000,2026-06-03,Air India,Chennai,Bangalore,80,On Time
F106,B1007,Arjun Verma,Economy,10000,2026-06-04,Akasa,Pune,Delhi,150,Delayed
F107,B1008,Meera Nair,Premium Economy,17000,2026-06-04,Vistara,Hyderabad,Kolkata,160,On Time
F108,B1009,Kiran Rao,Economy,9500,2026-06-05,Indigo,Mumbai,Hyderabad,110,On Time
F109,B1010,Nisha Reddy,Business,28000,2026-06-05,Akasa,Delhi,Chennai,145,Delayed


In [0]:
from pyspark.sql.window import Window

from pyspark.sql.functions import (
    sum,
    rank,
    dense_rank,
    row_number
)

In [0]:
flight_revenue = joined_df.groupBy(
    "flight_id"
).agg(
    sum("ticket_price").alias("revenue")
)

window_spec = Window.orderBy(
    flight_revenue["revenue"].desc()
)

top_flights = flight_revenue.withColumn(
    "rank",
    rank().over(window_spec)
)

display(
    top_flights.filter("rank <= 3")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


flight_id,revenue,rank
F101,39000,1
F103,38000,2
F109,28000,3


In [0]:
from pyspark.sql.functions import concat_ws

route_df = joined_df.withColumn(
    "route",
    concat_ws(
        " -> ",
        "from_city",
        "to_city"
    )
)

route_revenue = route_df.groupBy(
    "airline",
    "route"
).agg(
    sum("ticket_price").alias("revenue")
)

window_route = Window.partitionBy(
    "airline"
).orderBy(
    route_revenue["revenue"].desc()
)

top_routes = route_revenue.withColumn(
    "route_rank",
    row_number().over(window_route)
)

display(
    top_routes.filter(
        "route_rank = 1"
    )
)

airline,route,revenue,route_rank
Air India,Chennai -> Bangalore,25000,1
Akasa,Delhi -> Chennai,28000,1
Indigo,Hyderabad -> Delhi,39000,1
Vistara,Bangalore -> Hyderabad,38000,1


In [0]:
running_window = Window.orderBy(
    "booking_date"
)

running_revenue = df_bookings.withColumn(
    "running_revenue",
    sum("ticket_price").over(
        running_window
    )
)

display(running_revenue)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


booking_id,flight_id,passenger_name,travel_class,ticket_price,booking_date,running_revenue
B1001,F101,Rahul Sharma,Economy,8500,2026-06-01,30500
B1002,F101,Priya Reddy,Business,22000,2026-06-01,30500
B1003,F102,Amit Kumar,Economy,9000,2026-06-02,54500
B1004,F103,Sneha Patel,Premium Economy,15000,2026-06-02,54500
B1005,F104,Farhan Ali,Economy,7500,2026-06-03,87000
B1006,F105,Neha Singh,Business,25000,2026-06-03,87000
B1007,F106,Arjun Verma,Economy,10000,2026-06-04,114000
B1008,F107,Meera Nair,Premium Economy,17000,2026-06-04,114000
B1009,F108,Kiran Rao,Economy,9500,2026-06-05,151500
B1010,F109,Nisha Reddy,Business,28000,2026-06-05,151500


In [0]:
airline_revenue = joined_df.groupBy(
    "airline"
).agg(
    sum("ticket_price").alias("revenue")
)

airline_window = Window.orderBy(
    airline_revenue["revenue"].desc()
)

ranked_airlines = airline_revenue.withColumn(
    "airline_rank",
    rank().over(airline_window)
)

display(ranked_airlines)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


airline,revenue,airline_rank
Indigo,90000,1
Vistara,71500,2
Air India,68000,3
Akasa,62000,4


In [0]:
destination_df = joined_df.groupBy(
    "to_city"
).count()

destination_window = Window.orderBy(
    destination_df["count"].desc()
)

ranked_destinations = destination_df.withColumn(
    "dense_rank",
    dense_rank().over(
        destination_window
    )
)

display(ranked_destinations)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


to_city,count,dense_rank
Delhi,5,1
Hyderabad,4,2
Mumbai,3,3
Chennai,2,4
Bangalore,2,4
Kolkata,2,4
Goa,1,5
Pune,1,5
